# EPOCH 5th Datathon 본선 — 최종 제출 (blend_90, 검증 LB PR-AUC 0.7412)

## 개요

이 노트북은 본선 **확정 최고점 제출**인 `blend_90`(리더보드 PR-AUC **0.7412**, 한때 리더보드 1위)을 처음부터 끝까지 재현합니다.

## 접근 요약

두 개의 서로 다른 모델 출력을 **순위(rank) 기반으로 90:10 블렌딩**하여 각 단일 모델보다 높은 점수를 얻었습니다.

- **mate** — 팀원의 구조 시뮬레이션 모델(BEST_v2), 단독 리더보드 **0.7395**
- **q_joint** — 로지스틱 + 순위 변환 기반 내 모델, 단독 리더보드 **0.7115**

두 모델을 순위 변환 후 가중합(mate 0.90 : q_joint 0.10)하고, 그 결과를 다시 순위 변환하여 최종 **0.7412** 를 달성했습니다. PR-AUC/ROC-AUC 는 예측값의 순위에만 의존하므로, 순위를 보존하는 변환이면 점수가 동일하게 유지됩니다.

## 개선 과정

초기 2위(0.7075)에서 출발해, 입력 특징을 `lag1feat`(직전 시점 특징)과 `lag1q`(직전 시점 분위)로 보강하고 `q_joint`(결합 로지스틱) 모델로 개선하면서 단일 모델 성능을 끌어올렸습니다. 이후 구조적으로 다른 팀원의 시뮬레이션 모델(mate)과 순위 기반 블렌딩(90:10)으로 앙상블 다양성을 확보해 최종 0.7412(한때 리더보드 1위)에 도달했습니다. 이 노트북은 그 최종 산출물을 그대로 재현합니다.

## 1. 라이브러리 import 및 버전 확인

`pandas` 와 `numpy` 만 사용합니다. (로컬 환경: numpy < 2 권장)

In [ ]:
import pandas as pd
import numpy as np

print("pandas:", pd.__version__)
print("numpy :", np.__version__)

## 2. 입력 파일 로드

재현에 필요한 입력 파일 3개:

| 파일 | 위치 | 설명 |
|------|------|------|
| `sample_submission.csv` | `./epoch_data/` | 제출 양식 (row_id 순서 기준) |
| `mate.csv` | `./` | 팀원 BEST_v2 (LB 0.7395) |
| `submit_qjoint.csv` | `./` | 내 q_joint 모델 (LB 0.7115) |

컬럼명은 공백/BOM 을 제거하고, `sample_submission.csv` 의 `row_id` 순서로 정렬합니다.

In [ ]:
import os

# 경로 상수: sample_submission 은 epoch_data 폴더, 모델 출력은 현재 폴더
DATA_DIR = "./epoch_data"   # sample_submission.csv 위치
MODEL_DIR = "./"            # mate.csv / submit_qjoint.csv 위치

SS_PATH = os.path.join(DATA_DIR, "sample_submission.csv")
MATE_PATH = os.path.join(MODEL_DIR, "mate.csv")
QJOINT_PATH = os.path.join(MODEL_DIR, "submit_qjoint.csv")


def clean_cols(df):
    """컬럼명에서 공백과 BOM(\ufeff)을 제거."""
    df.columns = [c.strip().lstrip("\ufeff") for c in df.columns]
    return df


# sample_submission: 제출 순서(row_id)의 기준
ss = clean_cols(pd.read_csv(SS_PATH))
assert "row_id" in ss.columns, "sample_submission.csv 에 row_id 컬럼이 없습니다: %s" % list(ss.columns)
print("sample_submission.csv  rows=%d  cols=%s" % (len(ss), list(ss.columns)))

# mate (팀원 BEST_v2)
mate = clean_cols(pd.read_csv(MATE_PATH))
print("mate.csv               rows=%d  cols=%s" % (len(mate), list(mate.columns)))

# submit_qjoint (내 q_joint 모델)
qjoint = clean_cols(pd.read_csv(QJOINT_PATH))
print("submit_qjoint.csv      rows=%d  cols=%s" % (len(qjoint), list(qjoint.columns)))

# 제출 순서 기준 row_id 배열
order = ss.row_id.astype(str).values
N = len(order)
print("\n기준 행 수 N =", N)

## 3. 블렌딩 — blend_90 생성

`blend_all.py` 20번 줄 `mate × q_joint, w=0.90 → 0.7412(blend_90) 재현` 과 동일한 공식입니다.

1. 각 입력을 `sample_submission.csv` 의 row_id 순서로 reindex 한 뒤 `rank(pct=True)` 로 변환
2. 가중합: `score = 0.90 * rankpct(mate) + 0.10 * rankpct(qjoint)`
3. `score` 를 다시 `rank(pct=True)` 로 변환한 `rr`
4. 순위 보존 로지스틱: `prediction = 1 / (1 + exp(-8 * (rr - 0.8)))`

row_id 가 기준과 불일치하면 최저값으로 채웁니다.

In [ ]:
W_MATE = 0.90   # 주 모델(mate) 가중
W_QJOINT = 0.10  # 보조 모델(q_joint) 가중


def rankpct(df, label):
    """row_id 순서로 reindex 후 prediction 을 rank(pct=True) 배열로 변환."""
    assert "row_id" in df.columns and "prediction" in df.columns, \
        "%s 형식 이상(컬럼 %s)" % (label, list(df.columns))
    s = df.set_index(df.row_id.astype(str))["prediction"]
    s = s.reindex(order)
    if s.isna().any():
        miss = int(s.isna().sum())
        print("경고: %s 에서 %d개 row_id 가 sample_submission 과 불일치 -> 최저값으로 채움" % (label, miss))
        s = s.fillna(s.min() if s.notna().any() else 0.0)
    return s.rank(pct=True).values


# 1) 각 소스를 순위 변환
mate_r = rankpct(mate, "mate")
qjoint_r = rankpct(qjoint, "qjoint")

# 2) 가중합
score = W_MATE * mate_r + W_QJOINT * qjoint_r

# 3) 가중합 점수를 다시 순위 변환
rr = pd.Series(score).rank(pct=True).values

# 4) 순위 보존 로지스틱 변환으로 prediction 생성 (8자리 반올림)
pred = np.round(1.0 / (1.0 + np.exp(-8.0 * (rr - 0.8))), 8)

# 최종 제출 데이터프레임 (sample_submission 의 row_id 순서 유지)
blend_90 = ss[["row_id"]].copy()
blend_90["prediction"] = pred

# --- 중간값 분포 간단 출력 ---
def describe(name, arr):
    a = np.asarray(arr, dtype=float)
    print("  %-14s min=%.5f  mean=%.5f  max=%.5f" % (name, a.min(), a.mean(), a.max()))

print("[rankpct 분포]")
describe("rankpct(mate)", mate_r)
describe("rankpct(qjoint)", qjoint_r)
print("[score 분포]")
describe("score", score)
describe("rr(score rank)", rr)
describe("prediction", pred)

## 4. 저장 — blend_90_final.csv

694행, 컬럼 `row_id,prediction`, utf-8, LF 줄바꿈으로 저장합니다.

In [ ]:
OUT_PATH = "blend_90_final.csv"
blend_90.to_csv(OUT_PATH, index=False, encoding="utf-8", lineterminator="\n")
print("저장 완료:", os.path.abspath(OUT_PATH))
print("행 수:", len(blend_90), " 컬럼:", list(blend_90.columns))

## 5. 검증 — 저장된 파일 재로드 후 제출 형식 점검

저장된 `blend_90_final.csv` 를 다시 읽어 다음을 점검합니다 (OK / FAIL):

- 694행
- 컬럼 2개 (`row_id`, `prediction`)
- prediction 이 0~1 범위
- row_id 중복 0
- NaN 0

In [ ]:
chk = clean_cols(pd.read_csv(OUT_PATH))

checks = {
    "694행": len(chk) == 694,
    "컬럼 2개 (row_id, prediction)": list(chk.columns) == ["row_id", "prediction"],
    "prediction 0~1 범위": bool(chk["prediction"].between(0, 1).all()),
    "row_id 중복 0": int(chk["row_id"].duplicated().sum()) == 0,
    "NaN 0": int(chk.isna().sum().sum()) == 0,
}

all_ok = True
for name, passed in checks.items():
    status = "OK  " if passed else "FAIL"
    all_ok = all_ok and passed
    print("  [%s] %s" % (status, name))

print("\n전체 점검:", "OK — 제출 가능" if all_ok else "FAIL — 확인 필요")
print("prediction  min=%.8f  max=%.8f" % (chk["prediction"].min(), chk["prediction"].max()))

print("\n[상위 5개 prediction]")
print(chk.sort_values("prediction", ascending=False).head(5).to_string(index=False))
print("\n[하위 5개 prediction]")
print(chk.sort_values("prediction", ascending=True).head(5).to_string(index=False))

## 6. 마무리

이 노트북이 생성하는 **`blend_90_final.csv`** 는 리더보드 PR-AUC **0.7412** 제출과 **동일한 산출물**입니다 (`blend_90` = mate 0.90 : q_joint 0.10 순위 블렌딩).

### 재현 방법

필요한 입력 파일 3개:

1. `./epoch_data/sample_submission.csv` — 제출 양식(row_id 순서 기준)
2. `./mate.csv` — 팀원 BEST_v2 (LB 0.7395)
3. `./submit_qjoint.csv` — 내 q_joint 모델 (LB 0.7115)

실행 순서: 로컬 Jupyter(pandas, numpy < 2)에서 이 노트북의 셀을 **위에서 아래로 순서대로 실행**하면, 현재 폴더에 `blend_90_final.csv` 가 생성됩니다. 그 파일을 그대로 리더보드에 제출하면 됩니다.

> 참고: 순위 보존 변환(`rank(pct=True)` + 로지스틱)을 사용하므로 예측값은 0~1 범위의 확률 형태입니다. (순위값을 그대로 제출하는 형식은 과거 거부되었기 때문에 반드시 이 변환을 거칩니다.)